In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import os
import warnings
import pandas as pd
import spikeinterface.full as si
from pathlib import Path
from pprint import pprint
from tools.settings import settings
from tools.utils import find_raw_files, load_recording

# Ignore specific warnings from SpikeInterface
warnings.filterwarnings("ignore", message="The `ipywidgets` package is not installed.")
warnings.filterwarnings("ignore", message="`sortingview` is not installed.")

## 1. Set Paths and Parameters

In [13]:
# Load paths and experiment settings from configuration files
paths = settings.paths
experiment = settings.experiment

# Define base paths for data
raw_drive = paths.drive
experiment_name = experiment.dir
batch_folder = raw_drive / experiment_name / paths.data_dir
metadata_folder = raw_drive / experiment_name / paths.meta_dir
datasets_folder = raw_drive / experiment_name / paths.dataset_dir
print(f'Looking for recordings in:\n\t{batch_folder}')

# Define session paths
# raw_dir = paths.raw_dir
# output_dir = paths.output_dir

Looking for recordings in:
	/mnt/fasthd/3_TRAP_ISO/1_Recordings


#### Select recordings to process
The `recording_pairs` dictionary is loaded from your `run_config.toml` file.

In [14]:
recording_pairs = experiment.recordings
print("Found the following recordings to process:")
pprint(recording_pairs, indent=4, width=10)

Found the following recordings to process:
{   'TRP802_R1': {   'concatenate': False,
                     'multiple_shanks': True},
    'TRP802_R2': {   'concatenate': False,
                     'multiple_shanks': True}}


#### Set parallel processing parameters

In [15]:
job_kwargs = dict(
    n_jobs = 6, 
    chunk_duration = '1s', 
    progress_bar = True)
si.set_global_job_kwargs(**job_kwargs)

# Set compression and specific options for saving / writing recordings
from numcodecs import LZMA
compressor = LZMA()
saving_job_kwargs = dict(
    n_jobs=6,
    chunk_duration='1s',
    mp_context='spawn',  # linux only, otherwise use 'fork'
    max_threads_per_worker=4,
)

## 2. Postprocess Recordings
This section iterates through each recording and:
- Loads or creates an analyzer object for sortings
- Extracts waveforms, computes quality metrics and saves preliminary unit report
- Performs auto-merging and auto-curation using quality metrics
- Saves final unit report and optionally exports to IBL format for alignment
- Optionally, export to Phy for manual curation / inspection

In [16]:
# Set to True to skip recordings that have already been postprocessed
skip_existing = False
overwrite = True

# Define metrics to compute
metrics_list = [
    "amplitude_cutoff", "amplitude_median", "sliding_rp_violation",
    "presence_ratio", "snr", "isi_violation", "num_spikes",
    "peak_sign", "halfwidth", "peak_to_trough", "repolarization_slope",
    "recovery_slope",
]

## Load / create analyzer object and metrics

In [26]:
for session, properties in recording_pairs.items():
    animal = session.split('_')[0]
    recording_name = session
    concatenate = properties['concatenate']

    # session folder
    session_folder = batch_folder / animal / recording_name
    if session_folder.exists():
        print(f'recording session folder:  {session_folder}')  # top-level
    else:
        print(f'(!) No recording session folder found for:  {session_folder}\nSkipping...\n\n')
        continue
    
    print(f"--- Processing session: {recording_name} ---")

    # define I/O paths
    raw_dir = session_folder / paths.raw_dir
    processed_dir = session_folder / paths.processed_dir
    print(f'---saving processed outputs to: {processed_dir}---')
    
    # check for multiple probes
    match sortings := list(processed_dir.rglob(f'kilosort4_probe*')):
        case x if len(x) > 1:
            print(f'Found multiple probe sortings for {recording_name}: {x}')
        case _:
            print(f'Found single probe sorting for {recording_name}: {sortings}')
    
    for probe_num, sorter_output_folder in enumerate(sortings):
        print(f'\n----processing  probe {probe_num}{", concatenated..." if concatenate else ""}'.upper())

        analyzer_folder = processed_dir / f'sorting_analyzer_probe{probe_num}.zarr'

        # check if the session should be skipped
        if skip_existing and analyzer_folder.exists():
            print(f"Skipping {recording_name}: Postprocessed analyzer folder already exists.\n")
            continue

        if not sorter_output_folder.exists():
            print(f"Skipping {recording_name}: Sorter output not found at {sorter_output_folder}.\n")
            continue

        # ANALYZER OBJECT
        #   - load analyzer object if it already exists and not overwriting
        #   - if it doesn't exist, create it from preprocessed recording and sorting objects 
        if analyzer_folder.exists() and (overwrite is False):
            # load existing analyzer object
            analyzer = si.load_sorting_analyzer(
                folder=analyzer_folder,
                format="zarr"
            )
            print(f'Loaded existing analyzer from "{analyzer_folder}"...')
        else:
            # Load the sorted data
                # load sorting with all shanks aggregated
            keep_good = True  # only keep units labeled as 'good'
            # sortings should have been aggregated by shank during sorting
            assert sorter_output_folder.exists(), f"(!) Sorter output folder does not exist:\n\t{sorter_output_folder}\nSkipping...\n\n"
            try:
                print(f"Loading sorting output from: {sorter_output_folder}")
                if (agg_folder := sorter_output_folder / 'aggregated_sorting.zarr').exists():
                    aggregated_sorting = si.load(agg_folder)
                    print(aggregated_sorting, '\n')
                else:
                    raise FileNotFoundError(f"(!) Aggregated sorting file not found in {sorter_output_folder}")
            except FileNotFoundError as e:
                print(f"Could not load sorting output for {recording_name}. Error: {e}\n")
                continue

            # load preprocessed recording, or load and preprocess raw recording
            #   concatenates segments if needed
            preprocessed_rec_zarr = processed_dir / f'preprocessed_recording_probe{probe_num}.zarr'
            if preprocessed_rec_zarr.exists() and (not overwrite):
                print(f'Loading existing preprocessed recording from "{preprocessed_rec_zarr}"...')
                preprocessed_recording = si.load(preprocessed_rec_zarr)
            else:
                raw_folder = session_folder / raw_dir
                assert raw_folder.exists(), f"(!) No raw data folder found for recording: {session_folder}\nExpected in: {raw_folder}\nSkipping...\n\n"
                raw_files = find_raw_files(raw_folder, recording_name, concatenate=concatenate)
                if raw_files is None or len(raw_files) == 0:
                    print(f'(!) No raw data folder or files found for recording: {session_folder}\nExpected in: {raw_folder}\nSkipping...\n\n')
                    continue
                print(f'Loading raw recording:\n\t{raw_files[probe_num]}...')
                rec = load_recording(raw_files[probe_num], concatenate=concatenate) 
                if not rec:
                    print(f'(!) No valid recording found for {recording_name}!\nSkipping...\n\n')
                    continue
                split_recording_dict = rec.split_by("group")
                rec_filtered = si.bandpass_filter(split_recording_dict, freq_min=300.0, freq_max=6000.0)
                rec_filtered = si.phase_shift(recording=rec_filtered)
                preprocessed_recording = si.aggregate_channels(rec_filtered)
                bad_channel_ids, _ = si.detect_bad_channels(recording=preprocessed_recording)
                preprocessed_recording = si.interpolate_bad_channels(recording=preprocessed_recording, bad_channel_ids=bad_channel_ids)

                split_recording_dict = preprocessed_recording.split_by("group") # once more for spatial filtering
                try:
                    rec_filtered = si.highpass_spatial_filter(recording=split_recording_dict)
                except AssertionError:
                    rec_filtered = si.highpass_spatial_filter(recording=split_recording_dict, n_channel_pad=10)                        
                preprocessed_recording = si.aggregate_channels(rec_filtered)
                print('...finished preprocessing recording, saving to zarr...')
                preprocessed_recording = preprocessed_recording.save_to_zarr(
                    folder=preprocessed_rec_zarr, backend_options=dict(compressor=compressor), 
                    overwrite=True, verbose=True,
                    **saving_job_kwargs
                    )
            print('Preprocessed recording:\n\t', preprocessed_recording, '\n')
            aggregated_sorting.register_recording(preprocessed_recording)

            # clean up sorting, more curation later
            print('Cleaning up sorting...')
            print('...removing excess spikes...')
            aggregated_sorting = si.remove_excess_spikes(aggregated_sorting, preprocessed_recording)  # duplicated spikes
            print('...removing overlapping units...')
            aggregated_sorting = si.remove_redundant_units(aggregated_sorting, align=False, remove_strategy='max_spikes')  # overlapping units
            print('...removing units with too few spikes...')
            keep_units_idx = aggregated_sorting.count_num_spikes_per_unit('array') >= 500  # 500 spikes minimum for random_spikes
            keep_units = aggregated_sorting.get_unit_ids()[keep_units_idx]
            print(f'...keeping {len(keep_units)} units with at least 500 spikes.\n')
            aggregated_sorting = aggregated_sorting.select_units(keep_units)

            # create and save analyzer object to folder
            print(f'Creating analyzer in "{analyzer_folder}"...')
            analyzer = si.create_sorting_analyzer(
                sorting=aggregated_sorting,
                recording=preprocessed_recording,
                folder=analyzer_folder,
                format="zarr",  #"binary_folder", # for folder, zarr default for now
                sparse=True,
                backend_options=dict(compressor=compressor),
                overwrite=True,
            )     
        print(analyzer, '\n')

        # load or compute essential extensions first
        recompute_extensions = True
        print('Computing or loading essential extensions...')
        # specific parameters for these to avoid memory issues for these two steps
        for comp, param in {
        'random_spikes': dict(max_spikes_per_unit=300, seed=123), 
        'waveforms': {}
        }.items():
            if not analyzer.has_extension(comp) or recompute_extensions:
                print(f'...computing {comp} with parameters {param}' if (param != {}) else f'...computing {comp}')
                analyzer.compute(comp, **param, verbose=True)
            else:
                analyzer.load_extension(comp)
        print('Done.\n')

        # load or compute downstream extensions
        for comp in ['templates', 'noise_levels', 'isi_histograms', 'correlograms']:
            if not analyzer.has_extension(comp) or recompute_extensions:
                print(f'...computing {comp}')
                analyzer.compute(comp, verbose=True)
            else:
                print(f'...loading {comp}')
                analyzer.load_extension(comp)
        
        for comp, param in {
            'spike_amplitudes': {}, 
            'principal_components': dict(n_components=5, mode='by_channel_local'),
            'template_similarity': dict(method='cosine_similarity'), 
            'template_metrics': {}
            }.items():
            if not analyzer.has_extension(comp) or recompute_extensions:
                print(f'...computing {comp} with parameters {param}' if (param != {}) else f'...computing {comp}')
                analyzer.compute(comp, **param, verbose=True)
            else:
                analyzer.load_extension(comp)
        print('Done.\n')

        ## compute quality metrics
        metrics_output_folder = processed_dir / 'quality_metrics'
        metrics_output_folder.mkdir(exist_ok=True, parents=True)
        metrics_output_file = metrics_output_folder / f'preliminary_{recording_name}_unit_metrics_probe{probe_num}.csv'

        overwrite_metrics = True
        if metrics_output_file.exists() and (not overwrite_metrics):
            print(f'Loading existing metrics from {metrics_output_file}...')
            metrics = pd.read_csv(metrics_output_file)
        else:
            print('Computing unit quality metrics...')
            
            run_metrics = [
                'num_spikes', 'firing_rate',
                'amplitude_cutoff', 
                'presence_ratio', 
                'snr',
                'isi_violation', 'rp_violation',
                'd_prime',
                'isolation_distance'
            ]
            param_dict = {
                'num_spikes': {}, 
                'firing_rate': {}, 
                'presence_ratio': {'bin_duration_s': 180, 'mean_fr_ratio_thresh': 0.0}, 
                'snr': {'peak_sign': 'neg', 'peak_mode': 'extremum'}, 
                'isi_violation': {'isi_threshold_ms': 1.0, 'min_isi_ms': 0},
                'rp_violation': {'refractory_period_ms': 1.0, 'censored_period_ms': 0.0}, 
                'amplitude_cutoff': {'peak_sign': 'neg', 'num_histogram_bins': 100, 'histogram_smoothing_value': 3, 'amplitudes_bins_min_ratio': 5}, 
                # 'drift': {'interval_s': 60, 'min_spikes_per_interval': 10, 'direction': 'y', 'min_num_bins': 2},
                # 'nearest_neighbor': {'max_spikes': 1000, 'n_neighbors': 5},
                'isolation_distance':{},
                'd_prime':{}
                }
            # analyzer.compute('quality_metrics', metric_names=['nearest_neighbor'], save=False)
            qm_ext = analyzer.compute('quality_metrics', metric_names=run_metrics, metric_params=param_dict, verbose=True)
            print(f'...finished computing unit metrics, saving to {metrics_output_folder}...')
            metrics = qm_ext.get_data()
            metrics.to_csv(metrics_output_file, index=False)
            print('...finished saving unit metrics to CSV file.', '\n')

        # make preliminary summary plots of unit metrics
        print(f'Generating unit report for preliminary units...')
        report_folder = processed_dir / f'preliminary_unit_report_probe{probe_num}'
        si.export_report(analyzer, output_folder=report_folder, remove_if_exists=True)
        print(f'...finished generating report, saved to {report_folder}.\n')
        print(f'Done processing {recording_name}, probe {probe_num}.\n')
    print(f"--- Done processing {recording_name} ---\n\n")

recording session folder:  /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1
--- Processing session: TRP802_R1 ---
---saving processed outputs to: /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed---
Found single probe sorting for TRP802_R1: [PosixPath('/mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed/kilosort4_probe0')]

----PROCESSING  PROBE 0
Loading sorting output from: /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed/kilosort4_probe0
ZarrSortingExtractor: 378 units - 1 segments - 30.0kHz 

Found single raw file for TRP802_R1: [PosixPath('/mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/0_raw_compressed/TRP802_R1_g0_t0/TRP802_R1_g0_t0.imec0.ap.cbin')]
Loading raw recording:
	/mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/0_raw_compressed/TRP802_R1_g0_t0/TRP802_R1_g0_t0.imec0.ap.cbin...
...finished preprocessing recording, saving to zarr...
write_zarr_recording 
engine=process - n_jobs=6 - samples_per_chunk=29,979 - chunk_m

write_zarr_recording (workers: 6 processes):   0%|          | 0/6071 [00:00<?, ?it/s]

Preprocessed recording:
	 ZarrRecordingExtractor: 384 channels - 29979.845763 Hz - 1 segments - 181,993,829 samples 
                        6,070.54s (1.69 hours) - int16 dtype - 130.17 GiB 

Cleaning up sorting...
...removing excess spikes...
...removing overlapping units...
...removing units with too few spikes...
...keeping 378 units with at least 500 spikes.

Creating analyzer in "/mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed/sorting_analyzer_probe0.zarr"...


estimate_sparsity (workers: 6 processes):   0%|          | 0/6071 [00:00<?, ?it/s]

SortingAnalyzer: 384 channels - 378 units - 1 segments - zarr - sparse - has recording
Loaded 0 extensions 

Computing or loading essential extensions...
...computing random_spikes with parameters {'max_spikes_per_unit': 300, 'seed': 123}
...computing waveforms
compute_waveforms 
engine=process - n_jobs=6 - samples_per_chunk=29,979 - chunk_memory=21.96 MiB - total_memory=131.74 MiB - chunk_duration=1.00s (999.97 ms)


...

...computing template_similarity with parameters {'method': 'cosine_similarity'}
...computing template_metrics


/mnt/fasthd/01_Scripts/np-ephys/.venv/lib/python3.11/site-packages/spikeinterface/postprocessing/template_metrics.py:623: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  res = scipy.stats.linregress(times[peak_idx:max_idx], template_single[peak_idx:max_idx])


Done.

Computing unit quality metrics...
Computing num_spikes
Computing firing_rate
Computing amplitude_cutoff
Computing presence_ratio
Computing snr
Computing isi_violation
Computing rp_violation


calculate_pc_metrics:   0%|          | 0/545 [00:00<?, ?it/s]

...finished computing unit metrics, saving to /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2/2_processed/quality_metrics...
...finished saving unit metrics to CSV file. 

Generating unit report for preliminary units...
...finished generating report, saved to /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2/2_processed/preliminary_unit_report_probe0.

Done processing TRP802_R2, probe 0.

--- Done processing TRP802_R2 ---




### Auto-curation using unit quality metrics

In [27]:
export_to_alignment= True
export_to_phy = False

# quality metrics thresholds for filtering units
query = """
        num_spikes > 500 & \
        amplitude_cutoff < 0.1 & \
        isi_violations_ratio < 0.5 & \
        rp_contamination < 0.5 \
    """

In [29]:
for session, properties in recording_pairs.items():
    animal = session.split('_')[0]
    recording_name = session
    concatenate = properties['concatenate']

    # session folder
    session_folder = batch_folder / animal / recording_name
    if session_folder.exists():
        print(f'recording session folder:  {session_folder}')  # top-level
    else:
        print(f'(!) No recording session folder found for:  {session_folder}\nSkipping...\n\n')
        continue
    
    print(f"--- Processing session: {recording_name} ---")

    processed_dir = session_folder / paths.processed_dir
    alignment_dir = session_folder / paths.alignment_dir
    print(f'---saving processed outputs to: {processed_dir}---')

    probe_list = list(processed_dir.glob('sorting_analyzer_probe*'))  # check multiple probes, but not if multiple segments (ie, g0, g1 etc)
    if probe_list == []:
        probe_list = list(processed_dir.glob('sorting_analyzer_byShank*'))  # older name
    if probe_list != []:
        print(f"Found analyzers from {len(probe_list)} probes in {processed_dir}:\n\t", *[f.name for f in probe_list], sep='\n\t')
    assert probe_list!=[], f'No sorting analyzers found in {processed_dir}, check the folder!'

    for probe_num, analyzer_folder in enumerate(probe_list):
        print(f'\n----processing  probe {probe_num}{", concatenated..." if concatenate else ""}'.upper())

        # load analyzer object 
        # should be in: 
        #   analyzer_folder = processed_dir / f'sorting_analyzer_probe{probe_num}.zarr'
        assert analyzer_folder.exists(), f'----Analyzer folder not found: {analyzer_folder}'
        analyzer = si.load_sorting_analyzer(
            folder=analyzer_folder,
            format="zarr"
        )
        print(f'----Loading existing analyzer from {analyzer_folder}...')
        print(analyzer, '\n')

        ## load quality metrics
        metrics_output_folder = processed_dir / 'quality_metrics'
        metrics_output_file = metrics_output_folder / f'preliminary_{recording_name}_unit_metrics_probe{probe_num}.csv'
        try:
            assert metrics_output_file.exists(), f'Metrics file not found: {metrics_output_file}'
        except AssertionError:
            metrics_output_file = metrics_output_folder / f'{recording_name}_unit_metrics.csv'  # for older recordings
        finally:
            assert metrics_output_file.exists(), f'Metrics file not found: {metrics_output_file}'

        print(f'----Loading existing metrics from {metrics_output_file}...')
        metrics = pd.read_csv(metrics_output_file)

        # clean up sorting before merges
        print('----Cleaning up sorting before merges...')
        clean_sorting = si.remove_redundant_units(
            analyzer,
            align=True,
            remove_strategy="minimum_shift"
            )
        clean_sorting = si.remove_duplicated_spikes(
            clean_sorting, 
            method='keep_first_iterative'
            )
        # make clean copy of analyzer
        analyzer_clean = analyzer.select_units(clean_sorting.unit_ids)
        print('\t...Done.')

        # auto-merges
        print('----Auto-merging units...')
        template_diff_thresh = [0.05, 0.15, 0.25]
        presets = ["x_contaminations"] * len(template_diff_thresh)
        steps_params = [
            {"template_similarity": {"template_diff_thresh": i}}
            for i in template_diff_thresh
            ]
        try:
            analyzer_merged = si.auto_merge_units(
                analyzer_clean,
                presets=presets,
                steps_params=steps_params,
                recursive=True,
                merging_mode=True
                )
            print('\t...Done.')
        except:
            print('----Auto-merging failed, might need to inspect manually for merges...')
            analyzer_merged = analyzer_clean.copy()

        # filter units based on quality metrics
        metrics = analyzer_merged.get_extension('quality_metrics').get_data()
        keep_units = metrics.query(query)
        keep_unit_ids = keep_units.index.values  # unit ids to keep
        print(f'----Keeping {keep_unit_ids.shape[0]} units...')
        if keep_unit_ids.shape[0] == 0:
            print(f'No units left after filtering, exiting...   {recording_name}')
            continue
        analyzer_clean = analyzer_merged.select_units(
            keep_unit_ids, 
            format='zarr',
            folder=processed_dir / f'sorting_analyzer_clean_probe{probe_num}.zarr',
            )
        print('\t...Done.')
        # metrics.loc[keep_unit_ids,:]

        # make summary plots of filtered unit metrics
        report_folder = processed_dir / f'unit_report_clean_probe{probe_num}'
        print('----Generating unit report...')
        si.export_report(analyzer_clean, output_folder=report_folder, remove_if_exists=True)
        print(f'...finished generating unit report, saved to {report_folder}.\n')
        print('\t...Done.')

        # [optional] export to IBL GUI for alignment, if not done already
        export_to_alignment = True
        if export_to_alignment:
            ibl_folder = session_folder / '1_histology_alignment' / f'ibl_alignment_SI_probe{probe_num}'
            
            print('----Exporting to IBL GUI...')
            # optionally, we can pass an LFP recording to compute RMS/PSD in the LFP band
            recording_lfp = si.bandpass_filter(analyzer_clean.recording, freq_min=1, freq_max=300)
            # we can also decimate the LFP to speed up the process
            recording_lfp = si.decimate(recording_lfp, 10)
            
            # compute missing quality metrics for IBL alignment GUI
            analyzer_clean.compute('quality_metrics', metric_names=['amplitude_median'])
            si.export_to_ibl_gui(
                sorting_analyzer=analyzer_clean,
                output_folder=ibl_folder,
                lfp_recording=recording_lfp,
                remove_if_exists=True,
                verbose=True,
                # n_jobs=-1
            )
            print('\t...done.')
        
        # [optional] export to phy for final curation
        if export_to_phy:
            print('Exporting to phy...')
            si.export_to_phy(
                sorting_analyzer=analyzer_clean, 
                output_folder=processed_dir / 'phy_output', 
                remove_if_exists=True, 
                verbose=True,
                use_relative_path=True,
                **saving_job_kwargs
                )
            print('\t...Done.')
        print(f'\n----Done processing probe {recording_name} {probe_num}.')

    print(f'\n---Done processing  {recording_name}.\n\n')

recording session folder:  /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1
--- Processing session: TRP802_R1 ---
---saving processed outputs to: /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed---
Found analyzers from 1 probes in /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed:
	
	sorting_analyzer_probe0.zarr

----PROCESSING  PROBE 0
----Loading existing analyzer from /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed/sorting_analyzer_probe0.zarr...
SortingAnalyzer: 384 channels - 378 units - 1 segments - zarr - sparse - has recording
Loaded 12 extensions: correlograms, isi_histograms, noise_levels, principal_components, quality_metrics, random_spikes, spike_amplitudes, template_metrics, template_similarity, templates, unit_locations, waveforms 

----Loading existing metrics from /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R1/2_processed/quality_metrics/preliminary_TRP802_R1_unit_metrics_probe0.csv...
----Cleaning up sorting bef

compute_rms (workers: 6 processes):   0%|          | 0/2024 [00:00<?, ?it/s]

Computing LFP RMS
compute_rms 
engine=process - n_jobs=6 - samples_per_chunk=8,993 - chunk_memory=6.59 MiB - total_memory=39.52 MiB - chunk_duration=3.00s


compute_rms (workers: 6 processes):   0%|          | 0/2024 [00:00<?, ?it/s]

Computing LFP PSD
	...done.

----Done processing probe TRP802_R1 0.

---Done processing  TRP802_R1.


recording session folder:  /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2
--- Processing session: TRP802_R2 ---
---saving processed outputs to: /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2/2_processed---
Found analyzers from 1 probes in /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2/2_processed:
	
	sorting_analyzer_probe0.zarr

----PROCESSING  PROBE 0
----Loading existing analyzer from /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2/2_processed/sorting_analyzer_probe0.zarr...
SortingAnalyzer: 384 channels - 545 units - 1 segments - zarr - sparse - has recording
Loaded 12 extensions: correlograms, isi_histograms, noise_levels, principal_components, quality_metrics, random_spikes, spike_amplitudes, template_metrics, template_similarity, templates, unit_locations, waveforms 

----Loading existing metrics from /mnt/fasthd/3_TRAP_ISO/1_Recordings/TRP802/TRP802_R2/2_

compute_rms (workers: 6 processes):   0%|          | 0/2044 [00:00<?, ?it/s]

Computing LFP RMS
compute_rms 
engine=process - n_jobs=6 - samples_per_chunk=8,993 - chunk_memory=6.59 MiB - total_memory=39.52 MiB - chunk_duration=3.00s


compute_rms (workers: 6 processes):   0%|          | 0/2044 [00:00<?, ?it/s]

Computing LFP PSD
	...done.

----Done processing probe TRP802_R2 0.

---Done processing  TRP802_R2.


